# Telugu Scriptio Continua Word Segmentation (4-State `STATE_4` BIES Prediction) + CRF
## Abugida Syllable & Multi-Vottu Sub-Consonant Decomposition Framework with CRF Layer

This notebook trains sequence tagger models (**BiLSTM**, **BiGRU**, **BiRNN**, **CNN**) on the Telugu dataset to predict `STATE_4` BIES word boundaries (`B` = Begin, `I` = Inside, `E` = End, `S` = Single-character word) from continuous text, enhanced with a Conditional Random Field (CRF) layer to strictly enforce valid BIES transitions.

### Multi-Vottu Telugu Abugida Feature Representation
To handle complex Telugu/Sanskrit conjuncts with **multiple sub-consonants (వొత్తులు - Vottulu)** like `స్త్ర`, `స్ట్రా`, `డ్జ్ను`, `క్ష్మ`, each Telugu letter/Akshara is decomposed into **7 categorical feature slots**:
1. **Base Character / Consonant / Independent Vowel / Digit** (`base_id`)
2. **Virama (పొల్లు - Pollu)** (`virama_id`)
3. **Primary Sub-consonant (Vottu 1)** (`vottu1_id`)
4. **Secondary Sub-consonant (Vottu 2)** (`vottu2_id`)
5. **Tertiary Sub-consonant (Vottu 3)** (`vottu3_id`)
6. **Vowel Sign (మాత్ర - Maatra)** (`maatra_id`)
7. **Anusvara (సున్నా - Sunna)** / Visarga / Candrabindu (`sunna_id`)

Each component is embedded via a 64-dimensional trainable embedding layer, combined, and passed to the neural taggers, before finally passing through a CRF layer that enforces the BIES constraints and computes the optimal sequence using Viterbi Decoding.

### Kaggle Execution & Outputs Saved
- **Dataset Path**: `/kaggle/input/datasets/vallurikeerthiram/testing`
- **Saved Model PKLs**: `telugu_4state_crf_bilstm.pkl`, `telugu_4state_crf_gru.pkl`, `telugu_4state_crf_rnn.pkl`, `telugu_4state_crf_cnn.pkl`, `telugu_4state_crf_vocab.pkl`
- **Excel Output**: `telugu_4state_crf_predictions.xlsx`

In [1]:
# Step 1: Install required packages if needed
!pip install -q pandas openpyxl torch

In [2]:
from __future__ import annotations

import os
import random
import re
import sys
import warnings
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset

# ── Global Settings & Constants ───────────────────────────────────────────────
RANDOM_SEED = 42
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
NONE_TOKEN = "NONE"
PAD_LABEL = -100

LABEL_TO_ID = {"B": 0, "I": 1, "E": 2, "S": 3}
ID_TO_LABEL = {0: "B", 1: "I", 2: "E", 3: "S"}

AKSHARA_REGEX = re.compile(
    r'[\u0C05-\u0C39\u0C58-\u0C5A\u0030-\u0039A-Za-z]'
    r'[\u0C01-\u0C03\u0C3E-\u0C4C\u0C55\u0C56]*'
    r'(?:\u0C4D[\u0C15-\u0C39\u0C58-\u0C5A][\u0C01-\u0C03\u0C3E-\u0C4C\u0C55\u0C56]*)*'
    r'\u0C4D?'
    r'[\u0C01-\u0C03]?'
    r'|.'
)


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def select_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


@dataclass
class Config:
    data_dir: Path = Path("/kaggle/input/datasets/vallurikeerthiram/testing")
    local_fallback: Path = Path(".")
    excel_filename: str = "telugu_sentence_level_first_100_articles_split.xlsx"
    output_filename: str = "telugu_4state_crf_predictions.xlsx"
    embedding_dim: int = 64
    hidden_dim: int = 128
    cnn_channels: int = 128
    dropout: float = 0.2
    batch_size: int = 64
    epochs: int = 8
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    patience: int = 2
    num_workers: int = 0
    invalid_transition_penalty_weight: float = 2.0


set_seed(RANDOM_SEED)
device = select_device()
print(f"PyTorch version: {torch.__version__}")
print(f"Target Execution Device: {device}")
if device.type == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU Count: {torch.cuda.device_count()}")

PyTorch version: 2.10.0+cu128
Target Execution Device: cuda
GPU Name: Tesla T4
Total GPU Count: 2


In [3]:
# Step 2: Akshara Tokenization & Multi-Vottu Decomposition Logic
def extract_aksharas(text: str) -> List[str]:
    words = str(text).split("\u200c")
    aksharas: List[str] = []
    for word in words:
        if word:
            aksharas.extend(AKSHARA_REGEX.findall(word))
    return aksharas


def decompose_akshara(akshara_str: str) -> Tuple[str, str, str, str, str, str, str]:
    """Decomposes an Akshara into 7 feature slots:
    (base, virama, vottu1, vottu2, vottu3, maatra, sunna)
    """
    base = ""
    has_virama = NONE_TOKEN
    vottus: List[str] = []
    maatra = NONE_TOKEN
    sunna = NONE_TOKEN

    chars = list(akshara_str)
    if not chars:
        return (PAD_TOKEN, NONE_TOKEN, NONE_TOKEN, NONE_TOKEN, NONE_TOKEN, NONE_TOKEN, NONE_TOKEN)

    base = chars[0]
    i = 1
    n = len(chars)

    while i < n:
        c = chars[i]
        code = ord(c)
        if code == 0x0C4D:  # Virama ( Pollu )
            has_virama = "VIRAMA"
            i += 1
            if i < n and ((0x0C15 <= ord(chars[i]) <= 0x0C39) or (0x0C58 <= ord(chars[i]) <= 0x0C5A)):
                vottus.append(chars[i])  # Sub-consonant ( Vottu )
                i += 1
        elif (0x0C3E <= code <= 0x0C4C) or (code in (0x0C55, 0x0C56)):
            maatra = c  # Vowel Sign ( Maatra )
            i += 1
        elif 0x0C01 <= code <= 0x0C03:
            sunna = c  # Anusvara ( Sunna / Visarga )
            i += 1
        else:
            i += 1

    vottu1 = vottus[0] if len(vottus) > 0 else NONE_TOKEN
    vottu2 = vottus[1] if len(vottus) > 1 else NONE_TOKEN
    vottu3 = vottus[2] if len(vottus) > 2 else NONE_TOKEN

    return (base, has_virama, vottu1, vottu2, vottu3, maatra, sunna)


def parse_state4(state_text: object) -> List[int]:
    compact = "".join(str(state_text).strip().split()).upper()
    labels: List[int] = []
    for ch in compact:
        tag = "I" if ch == "M" else ch
        if tag in LABEL_TO_ID:
            labels.append(LABEL_TO_ID[tag])
    return labels


def reconstruct_sentence_from_state4(aksharas: Sequence[str], label_ids: Sequence[int]) -> str:
    words: List[str] = []
    curr: List[str] = []

    for ak, l_id in zip(aksharas, label_ids):
        tag = ID_TO_LABEL.get(int(l_id), "I")
        if tag == "S":
            if curr:
                words.append("".join(curr))
                curr = []
            words.append(ak)
        elif tag == "B":
            if curr:
                words.append("".join(curr))
            curr = [ak]
        elif tag == "I":
            curr.append(ak)
        elif tag == "E":
            curr.append(ak)
            words.append("".join(curr))
            curr = []
        else:
            curr.append(ak)

    if curr:
        words.append("".join(curr))

    return " ".join(words)

In [4]:
# Step 3: Vocabulary Management & Multi-Vottu Datasets
@dataclass
class TeluguVocab:
    base_vocab: Dict[str, int]
    virama_vocab: Dict[str, int]
    vottu_vocab: Dict[str, int]
    maatra_vocab: Dict[str, int]
    sunna_vocab: Dict[str, int]

    @classmethod
    def build(cls, train_df: pd.DataFrame) -> "TeluguVocab":
        base_counts = Counter()
        vottu_counts = Counter()
        maatra_counts = Counter()
        sunna_counts = Counter()

        for script in train_df["scriptio_continua"]:
            aks = extract_aksharas(script)
            for ak in aks:
                base, _, v1, v2, v3, maatra, sunna = decompose_akshara(ak)
                base_counts[base] += 1
                for v in (v1, v2, v3):
                    if v != NONE_TOKEN:
                        vottu_counts[v] += 1
                if maatra != NONE_TOKEN:
                    maatra_counts[maatra] += 1
                if sunna != NONE_TOKEN:
                    sunna_counts[sunna] += 1

        def make_dict(counts: Counter, extra_tokens: List[str]) -> Dict[str, int]:
            v = {PAD_TOKEN: 0, UNK_TOKEN: 1}
            for token in extra_tokens:
                if token not in v:
                    v[token] = len(v)
            for token in sorted(counts):
                if token not in v:
                    v[token] = len(v)
            return v

        base_vocab = make_dict(base_counts, [])
        virama_vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1, NONE_TOKEN: 2, "VIRAMA": 3}
        vottu_vocab = make_dict(vottu_counts, [NONE_TOKEN])
        maatra_vocab = make_dict(maatra_counts, [NONE_TOKEN])
        sunna_vocab = make_dict(sunna_counts, [NONE_TOKEN])

        return cls(base_vocab, virama_vocab, vottu_vocab, maatra_vocab, sunna_vocab)


class TeluguAksharaDataset(Dataset):
    def __init__(self, df: pd.DataFrame, vocab: TeluguVocab) -> None:
        self.dataframe = df.reset_index(drop=True)
        self.vocab = vocab

    def __len__(self) -> int:
        return len(self.dataframe)

    def __getitem__(self, idx: int) -> Dict[str, object]:
        row = self.dataframe.iloc[idx]
        script = str(row["scriptio_continua"])
        state4 = row["STATE4_LIST"]
        aksharas = extract_aksharas(script)

        min_len = min(len(aksharas), len(state4))
        aksharas = aksharas[:min_len]
        labels = state4[:min_len]

        base_ids, virama_ids, vottu1_ids, vottu2_ids, vottu3_ids, maatra_ids, sunna_ids = (
            [],
            [],
            [],
            [],
            [],
            [],
            [],
        )

        for ak in aksharas:
            base, virama, v1, v2, v3, maatra, sunna = decompose_akshara(ak)
            base_ids.append(self.vocab.base_vocab.get(base, self.vocab.base_vocab[UNK_TOKEN]))
            virama_ids.append(self.vocab.virama_vocab.get(virama, self.vocab.virama_vocab[UNK_TOKEN]))
            vottu1_ids.append(self.vocab.vottu_vocab.get(v1, self.vocab.vottu_vocab[UNK_TOKEN]))
            vottu2_ids.append(self.vocab.vottu_vocab.get(v2, self.vocab.vottu_vocab[UNK_TOKEN]))
            vottu3_ids.append(self.vocab.vottu_vocab.get(v3, self.vocab.vottu_vocab[UNK_TOKEN]))
            maatra_ids.append(self.vocab.maatra_vocab.get(maatra, self.vocab.maatra_vocab[UNK_TOKEN]))
            sunna_ids.append(self.vocab.sunna_vocab.get(sunna, self.vocab.sunna_vocab[UNK_TOKEN]))

        return {
            "base_ids": torch.tensor(base_ids, dtype=torch.long),
            "virama_ids": torch.tensor(virama_ids, dtype=torch.long),
            "vottu1_ids": torch.tensor(vottu1_ids, dtype=torch.long),
            "vottu2_ids": torch.tensor(vottu2_ids, dtype=torch.long),
            "vottu3_ids": torch.tensor(vottu3_ids, dtype=torch.long),
            "maatra_ids": torch.tensor(maatra_ids, dtype=torch.long),
            "sunna_ids": torch.tensor(sunna_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "art_id": row.get("Art_id", row.get("ART_ID", "")),
            "para_id": row.get("para_id", row.get("PARA_ID", "")),
            "sent_id": row.get("Sent_id", row.get("SENT_ID", "")),
            "scriptio_continua": script,
            "sentence_gt": str(row["Sentence"]),
            "aksharas": aksharas,
        }


def collate_telugu_batch(batch: Sequence[Dict[str, object]]) -> Dict[str, object]:
    padded_base = pad_sequence([b["base_ids"] for b in batch], batch_first=True, padding_value=0)
    padded_virama = pad_sequence([b["virama_ids"] for b in batch], batch_first=True, padding_value=0)
    padded_v1 = pad_sequence([b["vottu1_ids"] for b in batch], batch_first=True, padding_value=0)
    padded_v2 = pad_sequence([b["vottu2_ids"] for b in batch], batch_first=True, padding_value=0)
    padded_v3 = pad_sequence([b["vottu3_ids"] for b in batch], batch_first=True, padding_value=0)
    padded_maatra = pad_sequence([b["maatra_ids"] for b in batch], batch_first=True, padding_value=0)
    padded_sunna = pad_sequence([b["sunna_ids"] for b in batch], batch_first=True, padding_value=0)
    padded_labels = pad_sequence([b["labels"] for b in batch], batch_first=True, padding_value=PAD_LABEL)
    mask = padded_labels.ne(PAD_LABEL)

    return {
        "base_ids": padded_base,
        "virama_ids": padded_virama,
        "vottu1_ids": padded_v1,
        "vottu2_ids": padded_v2,
        "vottu3_ids": padded_v3,
        "maatra_ids": padded_maatra,
        "sunna_ids": padded_sunna,
        "labels": padded_labels,
        "mask": mask,
        "art_ids": [b["art_id"] for b in batch],
        "para_ids": [b["para_id"] for b in batch],
        "sent_ids": [b["sent_id"] for b in batch],
        "scriptio_continua": [b["scriptio_continua"] for b in batch],
        "sentence_gt": [b["sentence_gt"] for b in batch],
        "aksharas": [b["aksharas"] for b in batch],
    }

In [5]:
# Step 4: CRF Layer Implementation & BIES Constraints
class CRFLayer(nn.Module):
    def __init__(
        self,
        num_tags: int,
        transition_mask: torch.Tensor | None = None,
        start_mask: torch.Tensor | None = None,
        end_mask: torch.Tensor | None = None,
    ) -> None:
        super().__init__()
        self.start_transitions = nn.Parameter(torch.empty(num_tags))
        self.end_transitions = nn.Parameter(torch.empty(num_tags))
        self.transitions = nn.Parameter(torch.empty(num_tags, num_tags))
        self.register_buffer(
            "transition_mask",
            transition_mask if transition_mask is not None else torch.ones(num_tags, num_tags, dtype=torch.bool),
        )
        self.register_buffer(
            "start_mask",
            start_mask if start_mask is not None else torch.ones(num_tags, dtype=torch.bool),
        )
        self.register_buffer(
            "end_mask",
            end_mask if end_mask is not None else torch.ones(num_tags, dtype=torch.bool),
        )
        self.constraint_value = -1e4
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.uniform_(self.start_transitions, -0.1, 0.1)
        nn.init.uniform_(self.end_transitions, -0.1, 0.1)
        nn.init.uniform_(self.transitions, -0.1, 0.1)

    def _masked_start_transitions(self) -> torch.Tensor:
        return self.start_transitions.masked_fill(~self.start_mask, self.constraint_value)

    def _masked_end_transitions(self) -> torch.Tensor:
        return self.end_transitions.masked_fill(~self.end_mask, self.constraint_value)

    def _masked_transitions(self) -> torch.Tensor:
        return self.transitions.masked_fill(~self.transition_mask, self.constraint_value)

    def forward(self, emissions: torch.Tensor, tags: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        numerator = self._compute_score(emissions, tags, mask)
        denominator = self._compute_log_partition(emissions, mask)
        return torch.mean(denominator - numerator)

    def _compute_score(self, emissions: torch.Tensor, tags: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        start_transitions = self._masked_start_transitions()
        end_transitions = self._masked_end_transitions()
        transitions = self._masked_transitions()
        score = start_transitions[tags[:, 0]]
        score += emissions[:, 0, :].gather(1, tags[:, 0].unsqueeze(1)).squeeze(1)
        for timestep in range(1, emissions.size(1)):
            active = mask[:, timestep]
            prev_tags = tags[:, timestep - 1]
            curr_tags = tags[:, timestep]
            emit = emissions[:, timestep, :].gather(1, curr_tags.unsqueeze(1)).squeeze(1)
            score += (transitions[prev_tags, curr_tags] + emit) * active
        seq_ends = mask.long().sum(dim=1) - 1
        last_tags = tags.gather(1, seq_ends.unsqueeze(1)).squeeze(1)
        score += end_transitions[last_tags]
        return score

    def _compute_log_partition(self, emissions: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        start_transitions = self._masked_start_transitions()
        end_transitions = self._masked_end_transitions()
        transitions = self._masked_transitions()
        score = start_transitions + emissions[:, 0]
        for timestep in range(1, emissions.size(1)):
            next_score = score.unsqueeze(2) + transitions + emissions[:, timestep].unsqueeze(1)
            next_score = torch.logsumexp(next_score, dim=1)
            score = torch.where(mask[:, timestep].unsqueeze(1), next_score, score)
        score = score + end_transitions
        return torch.logsumexp(score, dim=1)

    def decode(self, emissions: torch.Tensor, mask: torch.Tensor) -> List[List[int]]:
        start_transitions = self._masked_start_transitions()
        end_transitions = self._masked_end_transitions()
        transitions = self._masked_transitions()
        score = start_transitions + emissions[:, 0]
        history: List[torch.Tensor] = []
        for timestep in range(1, emissions.size(1)):
            next_score = score.unsqueeze(2) + transitions
            best_score, best_path = next_score.max(dim=1)
            best_score = best_score + emissions[:, timestep]
            score = torch.where(mask[:, timestep].unsqueeze(1), best_score, score)
            history.append(best_path)
        score = score + end_transitions
        best_last_tags = score.argmax(dim=1)

        paths: List[List[int]] = []
        lengths = mask.long().sum(dim=1).tolist()
        for batch_idx, length in enumerate(lengths):
            tag = best_last_tags[batch_idx]
            path = [int(tag.item())]
            for hist in reversed(history[: max(length - 1, 0)]):
                tag = hist[batch_idx][tag]
                path.append(int(tag.item()))
            paths.append(list(reversed(path)))
        return paths


def build_bies_crf_constraints() -> Tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor]]:
    num_labels = len(LABEL_TO_ID)
    transition_mask = torch.zeros(num_labels, num_labels, dtype=torch.bool)
    start_mask = torch.zeros(num_labels, dtype=torch.bool)
    end_mask = torch.zeros(num_labels, dtype=torch.bool)

    valid_next = {
        "B": ("I", "E"),
        "I": ("I", "E"),
        "E": ("B", "S"),
        "S": ("B", "S"),
    }
    valid_start = ("B", "S")
    valid_end = ("E", "S")

    for label in valid_start:
        start_mask[LABEL_TO_ID[label]] = True
    for label in valid_end:
        end_mask[LABEL_TO_ID[label]] = True
    for current_label, next_labels in valid_next.items():
        for next_label in next_labels:
            transition_mask[LABEL_TO_ID[current_label], LABEL_TO_ID[next_label]] = True

    penalty_masks = {
        "transition": ~transition_mask,
        "start": ~start_mask,
        "end": ~end_mask,
    }
    crf_kwargs = {
        "transition_mask": transition_mask,
        "start_mask": start_mask,
        "end_mask": end_mask,
    }
    return crf_kwargs, penalty_masks


def compute_invalid_bies_penalty(
    emissions: torch.Tensor,
    mask: torch.Tensor,
    invalid_masks: Dict[str, torch.Tensor],
) -> torch.Tensor:
    probabilities = torch.softmax(emissions, dim=-1)
    lengths = mask.long().sum(dim=1)

    start_penalty = probabilities[:, 0, :][:, invalid_masks["start"].to(probabilities.device)].sum(dim=1)

    last_positions = (lengths - 1).clamp_min(0)
    last_probabilities = probabilities[torch.arange(probabilities.size(0), device=probabilities.device), last_positions]
    end_penalty = last_probabilities[:, invalid_masks["end"].to(probabilities.device)].sum(dim=1)

    transition_penalty = torch.zeros(probabilities.size(0), device=probabilities.device)
    transition_invalid = invalid_masks["transition"].to(probabilities.device)
    for timestep in range(1, probabilities.size(1)):
        active = mask[:, timestep].float()
        pair_probabilities = probabilities[:, timestep - 1].unsqueeze(2) * probabilities[:, timestep].unsqueeze(1)
        invalid_mass = pair_probabilities.masked_select(transition_invalid.unsqueeze(0)).view(probabilities.size(0), -1).sum(dim=1)
        transition_penalty = transition_penalty + invalid_mass * active

    normalizer = lengths.float().clamp_min(1.0)
    total_penalty = (start_penalty + end_penalty + transition_penalty) / normalizer
    return total_penalty.mean()

In [6]:
# Step 5: Multi-Vottu Feature Embedding & CRF Sequence Taggers
class TeluguAksharaEmbedding(nn.Module):
    def __init__(self, vocab: TeluguVocab, embed_dim: int = 64) -> None:
        super().__init__()
        self.emb_base = nn.Embedding(len(vocab.base_vocab), embed_dim, padding_idx=0)
        self.emb_virama = nn.Embedding(len(vocab.virama_vocab), embed_dim, padding_idx=0)
        self.emb_vottu = nn.Embedding(len(vocab.vottu_vocab), embed_dim, padding_idx=0)
        self.emb_maatra = nn.Embedding(len(vocab.maatra_vocab), embed_dim, padding_idx=0)
        self.emb_sunna = nn.Embedding(len(vocab.sunna_vocab), embed_dim, padding_idx=0)

        self.proj = nn.Linear(embed_dim * 7, embed_dim)
        self.act = nn.ReLU()

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        b_e = self.emb_base(batch["base_ids"])
        v_e = self.emb_virama(batch["virama_ids"])
        vt1_e = self.emb_vottu(batch["vottu1_ids"])
        vt2_e = self.emb_vottu(batch["vottu2_ids"])
        vt3_e = self.emb_vottu(batch["vottu3_ids"])
        m_e = self.emb_maatra(batch["maatra_ids"])
        s_e = self.emb_sunna(batch["sunna_ids"])

        concat = torch.cat([b_e, v_e, vt1_e, vt2_e, vt3_e, m_e, s_e], dim=-1)
        return self.act(self.proj(concat))


class SequenceTaggerCRFBase(nn.Module):
    def __init__(
        self,
        vocab: TeluguVocab,
        config: Config,
        num_labels: int = 4,
    ) -> None:
        super().__init__()
        self.embedding = TeluguAksharaEmbedding(vocab, config.embedding_dim)
        self.dropout = nn.Dropout(config.dropout)
        
        crf_kwargs, invalid_penalty_masks = build_bies_crf_constraints()
        self.crf = CRFLayer(num_labels, **crf_kwargs)
        self.invalid_penalty_masks = invalid_penalty_masks
        self.invalid_penalty_weight = config.invalid_transition_penalty_weight

    def loss(self, batch: Dict[str, torch.Tensor], tags: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        emissions = self.get_emissions(batch)
        safe_tags = tags.masked_fill(~mask, 0)
        loss = self.crf(emissions, safe_tags, mask)
        
        if self.invalid_penalty_masks and self.invalid_penalty_weight > 0:
            loss = loss + self.invalid_penalty_weight * compute_invalid_bies_penalty(
                emissions, mask, self.invalid_penalty_masks
            )
        return loss

    def decode(self, batch: Dict[str, torch.Tensor], mask: torch.Tensor) -> List[List[int]]:
        emissions = self.get_emissions(batch)
        return self.crf.decode(emissions, mask)
        
    def get_emissions(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        raise NotImplementedError


class BiLSTMTaggerCRF(SequenceTaggerCRFBase):
    def __init__(self, vocab: TeluguVocab, config: Config) -> None:
        super().__init__(vocab, config)
        self.lstm = nn.LSTM(
            input_size=config.embedding_dim,
            hidden_size=config.hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.classifier = nn.Linear(config.hidden_dim * 2, 4)

    def get_emissions(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        x = self.embedding(batch)
        outputs, _ = self.lstm(self.dropout(x))
        return self.classifier(self.dropout(outputs))


class GRUTaggerCRF(SequenceTaggerCRFBase):
    def __init__(self, vocab: TeluguVocab, config: Config) -> None:
        super().__init__(vocab, config)
        self.gru = nn.GRU(
            input_size=config.embedding_dim,
            hidden_size=config.hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.classifier = nn.Linear(config.hidden_dim * 2, 4)

    def get_emissions(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        x = self.embedding(batch)
        outputs, _ = self.gru(self.dropout(x))
        return self.classifier(self.dropout(outputs))


class RNNTaggerCRF(SequenceTaggerCRFBase):
    def __init__(self, vocab: TeluguVocab, config: Config) -> None:
        super().__init__(vocab, config)
        self.rnn = nn.RNN(
            input_size=config.embedding_dim,
            hidden_size=config.hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
            nonlinearity="tanh",
        )
        self.classifier = nn.Linear(config.hidden_dim * 2, 4)

    def get_emissions(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        x = self.embedding(batch)
        outputs, _ = self.rnn(self.dropout(x))
        return self.classifier(self.dropout(outputs))


class CNNTaggerCRF(SequenceTaggerCRFBase):
    def __init__(self, vocab: TeluguVocab, config: Config) -> None:
        super().__init__(vocab, config)
        self.conv = nn.Conv1d(config.embedding_dim, config.cnn_channels, kernel_size=3, padding=1)
        self.activation = nn.ReLU()
        self.classifier = nn.Linear(config.cnn_channels, 4)

    def get_emissions(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        x = self.dropout(self.embedding(batch)).transpose(1, 2)
        x = self.activation(self.conv(x)).transpose(1, 2)
        return self.classifier(self.dropout(x))


def build_model(model_name: str, vocab: TeluguVocab, config: Config) -> nn.Module:
    name = model_name.lower()
    if name == "bilstm":
        return BiLSTMTaggerCRF(vocab, config)
    if name == "gru":
        return GRUTaggerCRF(vocab, config)
    if name == "rnn":
        return RNNTaggerCRF(vocab, config)
    if name == "cnn":
        return CNNTaggerCRF(vocab, config)
    raise ValueError(f"Unknown model name: {model_name}")

In [7]:
# Step 6: CRF Training & Viterbi Inference Functions
def train_one_epoch(
    model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer, device: torch.device
) -> float:
    model.train()
    total_loss = 0.0

    for batch in loader:
        b_dev = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
        labels = b_dev["labels"]
        mask = b_dev["mask"]
        
        optimizer.zero_grad()
        # model.module if DataParallel is used
        raw_model = model.module if isinstance(model, nn.DataParallel) else model
        loss = raw_model.loss(b_dev, labels, mask)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / max(len(loader), 1)


def predict_and_generate_rows(
    model: nn.Module,
    loader: DataLoader,
    device: torch.device,
    model_name: str,
    split_name: str,
) -> pd.DataFrame:
    model.eval()
    rows: List[Dict[str, object]] = []

    with torch.no_grad():
        for batch in loader:
            b_dev = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}
            labels = b_dev["labels"]
            mask = b_dev["mask"]

            raw_model = model.module if isinstance(model, nn.DataParallel) else model
            predicted_paths = raw_model.decode(b_dev, mask)

            batch_size_actual = len(batch["art_ids"])
            for idx in range(batch_size_actual):
                seq_mask = mask[idx]
                true_seq = labels[idx][seq_mask].tolist()
                pred_seq = predicted_paths[idx]

                aksharas = batch["aksharas"][idx][: len(true_seq)]
                script = batch["scriptio_continua"][idx]
                gt_sentence = batch["sentence_gt"][idx]

                gt_labels_str = "".join(ID_TO_LABEL.get(x, "I") for x in true_seq)
                pred_labels_str = "".join(ID_TO_LABEL.get(x, "I") for x in pred_seq)

                predicted_sentence = reconstruct_sentence_from_state4(aksharas, pred_seq)

                rows.append(
                    {
                        "Art_id": batch["art_ids"][idx],
                        "para_id": batch["para_ids"][idx],
                        "Sent_id": batch["sent_ids"][idx],
                        "INPUT_SENTENCE": script,
                        "GROUND_TRUTH_SENTENCE": gt_sentence,
                        "PREDICTED_SENTENCE": predicted_sentence,
                        "GROUND_TRUTH_LABELS": gt_labels_str,
                        "PREDICTED_LABELS": pred_labels_str,
                        "MODEL": model_name.upper(),
                        "SPLIT": split_name.upper(),
                    }
                )

    return pd.DataFrame(rows)


def fit_and_evaluate_model(
    model_name: str,
    vocab: TeluguVocab,
    loaders: Dict[str, DataLoader],
    device: torch.device,
    config: Config,
) -> Dict[str, pd.DataFrame]:
    print(f"\n================ Training {model_name.upper()} + CRF ================")
    model = build_model(model_name, vocab, config).to(device)

    if torch.cuda.device_count() > 1:
        print(f"[Multi-GPU] Wrapping {model_name.upper()}-CRF with DataParallel across {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)

    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)

    for epoch in range(1, config.epochs + 1):
        loss = train_one_epoch(model, loaders["train"], optimizer, device)
        print(f"[{model_name.upper()}-CRF] Epoch {epoch}/{config.epochs} | Train Loss: {loss:.4f}")

    # ── Save Model PKL Checkpoint File ─────────────────────────────────────────
    raw_model = model.module if isinstance(model, nn.DataParallel) else model
    pkl_path = Path(f"telugu_4state_crf_{model_name.lower()}.pkl")
    torch.save(
        {
            "model_name": model_name.upper() + "_CRF",
            "scheme": "4STATE",
            "model_state_dict": raw_model.state_dict(),
            "vocab": vocab,
        },
        pkl_path,
    )
    print(f"Saved trained model PKL checkpoint: {pkl_path.resolve()}")

    results = {}
    for split_name in ("train", "val", "test"):
        df_pred = predict_and_generate_rows(model, loaders[split_name], device, model_name, split_name)
        results[split_name] = df_pred

    return results

In [8]:
# Step 7: Dataset Loading, Model Execution & Output Generation
config = Config()

search_paths = [
    config.data_dir / config.excel_filename,
    Path("/kaggle/input/datasets/vallurikeerthiram/testing") / config.excel_filename,
    Path("c:/Users/keert/OneDrive - Amrita vishwa vidyapeetham/3 Phrase Project/phase 2/telugu/training") / config.excel_filename,
    Path(config.excel_filename)
]

target_path = None
for p in search_paths:
    if p.exists():
        target_path = p
        break

if target_path is None:
    for root, _, files in os.walk("/kaggle/input"):
        for f in files:
            if f.endswith(".xlsx"):
                target_path = Path(root) / f
                break
        if target_path:
            break

if target_path is None or not target_path.exists():
    raise FileNotFoundError("Dataset Excel file not found. Please verify Kaggle input dataset path.")

print(f"Successfully located dataset Excel file at: {target_path.resolve()}")
wb = pd.ExcelFile(target_path)

sheet_map = {}
for s in wb.sheet_names:
    s_lower = s.lower()
    if "train" in s_lower:
        sheet_map["train"] = s
    elif "val" in s_lower:
        sheet_map["val"] = s
    elif "test" in s_lower:
        sheet_map["test"] = s

splits_df: Dict[str, pd.DataFrame] = {}
for key, sname in sheet_map.items():
    df = wb.parse(sname)
    df["STATE4_LIST"] = df["4state"].map(parse_state4)
    valid_rows = [len(r["STATE4_LIST"]) > 0 for _, r in df.iterrows()]
    splits_df[key] = df.loc[valid_rows].reset_index(drop=True)

print("Building Multi-Vottu 4-State Telugu Abugida vocabularies...")
vocab = TeluguVocab.build(splits_df["train"])

# ── Save Vocabulary PKL File ───────────────────────────────────────────────
vocab_path = Path("telugu_4state_crf_vocab.pkl")
torch.save(vocab, vocab_path)
print(f"Saved vocabulary file: {vocab_path.resolve()}")

loaders: Dict[str, DataLoader] = {}
for split_name, df_split in splits_df.items():
    ds = TeluguAksharaDataset(df_split, vocab)
    loaders[split_name] = DataLoader(
        ds,
        batch_size=config.batch_size,
        shuffle=(split_name == "train"),
        num_workers=config.num_workers,
        collate_fn=collate_telugu_batch,
    )

all_models = ["bilstm", "gru", "rnn", "cnn"]
split_results: Dict[str, List[pd.DataFrame]] = {"train": [], "val": [], "test": []}

for model_name in all_models:
    model_dfs = fit_and_evaluate_model(model_name, vocab, loaders, device, config)
    for sname in ("train", "val", "test"):
        split_results[sname].append(model_dfs[sname])

out_file = Path(config.output_filename)
print(f"\nWriting predictions to Excel output file: {out_file.resolve()}")

with pd.ExcelWriter(out_file, engine="openpyxl") as writer:
    for sname in ("train", "val", "test"):
        combined_df = pd.concat(split_results[sname], ignore_index=True)
        sheet_title = f"{sname.upper()}_PREDICTIONS"[:31]
        combined_df.to_excel(writer, sheet_name=sheet_title, index=False)

print(f"\nMulti-Vottu 4-State CRF Training & PKL Export completed successfully!")

Successfully located dataset Excel file at: /kaggle/input/datasets/abhaygokavarapu0630/telugu100/telugu_sentence_level_first_100_articles_split.xlsx
Building Multi-Vottu 4-State Telugu Abugida vocabularies...
Saved vocabulary file: /kaggle/working/telugu_4state_crf_vocab.pkl

================ Training BILSTM + CRF ================
[Multi-GPU] Wrapping BILSTM-CRF with DataParallel across 2 GPUs
[BILSTM-CRF] Epoch 1/8 | Train Loss: 225.4807
[BILSTM-CRF] Epoch 2/8 | Train Loss: 222.3211
[BILSTM-CRF] Epoch 3/8 | Train Loss: 271.5613
[BILSTM-CRF] Epoch 4/8 | Train Loss: 322.4397
[BILSTM-CRF] Epoch 5/8 | Train Loss: 216.5564
[BILSTM-CRF] Epoch 6/8 | Train Loss: 215.6502
[BILSTM-CRF] Epoch 7/8 | Train Loss: 215.1742
[BILSTM-CRF] Epoch 8/8 | Train Loss: 266.6652
Saved trained model PKL checkpoint: /kaggle/working/telugu_4state_crf_bilstm.pkl

================ Training GRU + CRF ================
[Multi-GPU] Wrapping GRU-CRF with DataParallel across 2 GPUs
[GRU-CRF] Epoch 1/8 | Train Loss: 225.3